In [0]:
# ── NOTEBOOK 1: Bronze Layer ──────────────────────────────────────────────
# Read the raw Delta table created from the NYC Taxi parquet upload

df_bronze = spark.table("workspace.default.bronze_taxi_raw")

print(f"✅ Bronze table loaded")
print(f"   Rows    : {df_bronze.count():,}")
print(f"   Columns : {len(df_bronze.columns)}")
df_bronze.printSchema()

✅ Bronze table loaded
   Rows    : 3,066,766
   Columns : 19
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [0]:
# ── NOTEBOOK 2: Silver Layer — Cleaned & Validated ───────────────────────
# Apply cleaning rules, cast types, remove bad data, add derived columns.
# Silver = trusted, cleaned data ready for aggregation.

from pyspark.sql.functions import (
    col, hour, dayofweek, month, when, round as spark_round,
    to_date, datediff, unix_timestamp
)

df_silver = (
    df_bronze
    # Remove nulls in critical columns
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    .filter(col("fare_amount").isNotNull())
    .filter(col("trip_distance").isNotNull())

    # Remove bad data (business rules)
    .filter(col("fare_amount") > 0)
    .filter(col("trip_distance") > 0)
    .filter(col("passenger_count") > 0)
    .filter(col("passenger_count") <= 6)
    .filter(col("trip_distance") <= 100)
    .filter(col("fare_amount") <= 500)

    # Filter to only valid 2023 January dates
    .filter(col("tpep_pickup_datetime") >= "2023-01-01")
    .filter(col("tpep_pickup_datetime") <  "2023-02-01")

    # Select and rename columns
    .select(
        col("VendorID").alias("vendor_id"),
        col("tpep_pickup_datetime").alias("pickup_datetime"),
        col("tpep_dropoff_datetime").alias("dropoff_datetime"),
        col("passenger_count").cast("integer").alias("passenger_count"),
        col("trip_distance"),
        col("PULocationID").alias("pickup_location_id"),
        col("DOLocationID").alias("dropoff_location_id"),
        col("payment_type"),
        col("fare_amount"),
        col("tip_amount"),
        col("tolls_amount"),
        col("total_amount"),
        col("congestion_surcharge"),
        col("airport_fee"),
    )

    # Add derived columns
    .withColumn("pickup_date",    to_date(col("pickup_datetime")))
    .withColumn("pickup_hour",    hour(col("pickup_datetime")))
    .withColumn("pickup_dow",     dayofweek(col("pickup_datetime")))
    .withColumn("trip_duration_mins",
        spark_round(
            (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 60,
            1
        )
    )
    .withColumn("payment_method",
        when(col("payment_type") == 1, "Credit card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No charge")
        .when(col("payment_type") == 4, "Dispute")
        .otherwise("Unknown")
    )
    .withColumn("tip_pct",
        spark_round(col("tip_amount") / col("fare_amount") * 100, 1)
    )

    # Remove trips with negative duration
    .filter(col("trip_duration_mins") > 0)
    .filter(col("trip_duration_mins") <= 180)
)

print(f"Silver layer cleaned")
print(f"   Bronze rows : 3,066,766")
print(f"   Silver rows : {df_silver.count():,}")
print(f"   Removed     : {3066766 - df_silver.count():,} bad rows")

# Write to Delta table
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_taxi_cleaned")

print(f"Saved to workspace.default.silver_taxi_cleaned")

Silver layer cleaned
   Bronze rows : 3,066,766
   Silver rows : 2,880,923
   Removed     : 185,843 bad rows
Saved to workspace.default.silver_taxi_cleaned


In [0]:
# ── NOTEBOOK 3: Gold Layer — Analytics Aggregations ──────────────────────
# Build 4 Gold tables from Silver,ready for dashboards and BI queries.

from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    max as spark_max, min as spark_min
)

df_silver = spark.table("workspace.default.silver_taxi_cleaned")

# ── Gold Table 1: Revenue by hour of day ─────────────────────────────────
gold_hourly = (
    df_silver
    .groupBy("pickup_hour")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("total_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance"),
        spark_round(avg("tip_pct"), 2).alias("avg_tip_pct"),
    )
    .orderBy("pickup_hour")
)

gold_hourly.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_hourly_demand")
print("gold_hourly_demand saved")

# ── Gold Table 2: Revenue by day of week ─────────────────────────────────
gold_daily = (
    df_silver
    .groupBy("pickup_dow")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("total_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_duration_mins"), 2).alias("avg_duration_mins"),
    )
    .withColumn("day_name",
        col("pickup_dow").cast("string")
    )
    .orderBy("pickup_dow")
)

gold_daily.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_demand")
print("gold_daily_demand saved")

# ── Gold Table 3: Payment method analysis ────────────────────────────────
gold_payment = (
    df_silver
    .groupBy("payment_method")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("tip_amount"), 2).alias("avg_tip"),
        spark_round(avg("tip_pct"), 2).alias("avg_tip_pct"),
    )
    .orderBy(col("total_trips").desc())
)

gold_payment.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_payment_analysis")
print("gold_payment_analysis saved")

# ── Gold Table 4: Top pickup locations ───────────────────────────────────
gold_locations = (
    df_silver
    .groupBy("pickup_location_id")
    .agg(
        count("*").alias("total_trips"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance"),
        spark_round(avg("total_amount"), 2).alias("avg_fare"),
    )
    .orderBy(col("total_trips").desc())
    .limit(50)
)

gold_locations.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.gold_top_locations")
print("gold_top_locations saved")

print("\nAll 4 Gold tables saved successfully")
print(f"   gold_hourly_demand    — demand and revenue by hour")
print(f"   gold_daily_demand     — demand and revenue by day of week")
print(f"   gold_payment_analysis — payment method breakdown")
print(f"   gold_top_locations    — top 50 pickup locations")

gold_hourly_demand saved
gold_daily_demand saved
gold_payment_analysis saved
gold_top_locations saved

All 4 Gold tables saved successfully
   gold_hourly_demand    — demand and revenue by hour
   gold_daily_demand     — demand and revenue by day of week
   gold_payment_analysis — payment method breakdown
   gold_top_locations    — top 50 pickup locations


In [0]:
# ── NOTEBOOK 4: Delta Lake Features Demo ─────────────────────────────────
# Demonstrates ACID transactions, time travel, and schema evolution.
# These are the features that make Delta Lake different from plain parquet.

# ── 1. ACID Transactions — show Delta transaction log ────────────────────
print("=" * 55)
print("DELTA LAKE FEATURES DEMONSTRATION")
print("=" * 55)

# Show Delta table history (transaction log)
print("\n1. TRANSACTION LOG (ACID Transactions)")
print("   Every write to a Delta table is logged:")
spark.sql("DESCRIBE HISTORY workspace.default.silver_taxi_cleaned") \
    .select("version", "timestamp", "operation", "operationParameters") \
    .show(5, truncate=False)

# ── 2. Time Travel — query previous versions ─────────────────────────────
print("\n2. TIME TRAVEL")
print("   Query the table AS OF version 0 (original write):")
df_v0 = spark.sql("""
    SELECT COUNT(*) as row_count
    FROM workspace.default.silver_taxi_cleaned VERSION AS OF 0
""")
df_v0.show()

# ── 3. Schema enforcement — Delta rejects bad schema ─────────────────────
print("\n3. SCHEMA ENFORCEMENT")
print("   Attempting to write wrong schema → Delta rejects it:")
from pyspark.sql import Row
bad_df = spark.createDataFrame([Row(wrong_column="test", bad_data=999)])
try:
    bad_df.write.format("delta").mode("append") \
        .saveAsTable("workspace.default.silver_taxi_cleaned")
    print(" Should have failed!")
except Exception as e:
    print(f"Delta correctly rejected bad schema: schema mismatch detected")

# ── 4. Delta table stats ──────────────────────────────────────────────────
print("\n4. TABLE STATISTICS")
spark.sql("DESCRIBE DETAIL workspace.default.silver_taxi_cleaned") \
    .select("format", "numFiles", "sizeInBytes", "location") \
    .show(truncate=False)

# ── 5. Summary ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("MEDALLION ARCHITECTURE SUMMARY")
print("=" * 55)

bronze = spark.table("workspace.default.bronze_taxi_raw").count()
silver = spark.table("workspace.default.silver_taxi_cleaned").count()

print(f"""
Dataset  : NYC Yellow Taxi — January 2023
Source   : TLC Trip Record Data (public)

Layer          Rows        Table
──────────────────────────────────────────────────
Bronze         {bronze:>10,}   bronze_taxi_raw
Silver         {silver:>10,}   silver_taxi_cleaned
Gold (hourly)          24   gold_hourly_demand
Gold (daily)            7   gold_daily_demand
Gold (payment)          5   gold_payment_analysis
Gold (locations)       50   gold_top_locations

Format: Delta Lake (Parquet + transaction log)
""")

DELTA LAKE FEATURES DEMONSTRATION

1. TRANSACTION LOG (ACID Transactions)
   Every write to a Delta table is logged:
+-------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationParameters                                                                                                                                                                                                                                                                                                            |
+-------+-------------------+---------------------------------+----------------------------------------------------